# MMM Stress Test #7 — Baseline Flexibility
**Question:** Can an overly flexible baseline absorb true media effects — or can an inflexible one force media to explain baseline movement?

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

## Section 1 — Config

In [ ]:
N_WEEKS = 156
TRUE_ROAS = {"meta": 2.0, "search": 5.0, "tv": 1.5}
NOISE_SD = 60_000
RHO = 0.10
N_SIMS = 200

print("Weeks:", N_WEEKS)
print("True ROAS:", TRUE_ROAS)
print("Noise SD:", f"\u00a3{NOISE_SD:,.0f}")
print("Simulations:", N_SIMS)

## Section 2 — Data Generating Process

The DGP includes a **medium-frequency demand component** (26-week cycle) in the baseline but this component is NOT included as a feature in Model A or Model B. This is the key stress: models that don't control for it may misattribute demand variation to media spend.

In [ ]:
def generate_data(seed=42, n_weeks=N_WEEKS, noise_sd=NOISE_SD, rho=RHO):
    rng = np.random.default_rng(seed)
    t = np.arange(n_weeks)

    promo = rng.binomial(1, 0.12, n_weeks)
    sin52 = np.sin(2 * np.pi * t / 52)
    cos52 = np.cos(2 * np.pi * t / 52)
    medium_freq = 300_000 * np.sin(2 * np.pi * t / 26)

    baseline = (
        2_000_000
        + 4_000 * t
        + 500_000 * sin52
        + 150_000 * cos52
        + medium_freq  # in DGP but not in Model A or B
        + 400_000 * promo
    )

    covariance = np.array([[1.0, rho], [rho, 1.0]])
    shocks = rng.multivariate_normal(mean=[0, 0], cov=covariance, size=n_weeks)

    meta_spend = np.clip(55_000 + 22_000 * shocks[:, 0], 5_000, None)
    search_spend = np.clip(42_000 + 16_000 * shocks[:, 1], 5_000, None)
    tv_spend = np.clip(90_000 + 45_000 * rng.normal(size=n_weeks), 0, None)

    revenue = (
        baseline
        + TRUE_ROAS["meta"] * meta_spend
        + TRUE_ROAS["search"] * search_spend
        + TRUE_ROAS["tv"] * tv_spend
        + rng.normal(0, noise_sd, n_weeks)
    )

    # Precompute all harmonics for Model C
    sin26 = np.sin(2 * np.pi * t / 26)
    cos26 = np.cos(2 * np.pi * t / 26)
    sin13 = np.sin(2 * np.pi * t / 13)
    cos13 = np.cos(2 * np.pi * t / 13)

    return pd.DataFrame({
        "week": t,
        "revenue": revenue,
        "baseline_true": baseline,
        "meta_spend": meta_spend,
        "search_spend": search_spend,
        "tv_spend": tv_spend,
        "promo": promo,
        "trend": t,
        "sin52": sin52,
        "cos52": cos52,
        "sin26": sin26,
        "cos26": cos26,
        "sin13": sin13,
        "cos13": cos13,
    })

## Section 3 — Inspect One Dataset

In [ ]:
df = generate_data(seed=42)
print("Dataset shape:", df.shape)
display(df.head())

truth_table = pd.DataFrame({
    "channel": ["meta", "search", "tv"],
    "true_roas": [TRUE_ROAS["meta"], TRUE_ROAS["search"], TRUE_ROAS["tv"]],
    "total_spend": [df["meta_spend"].sum(), df["search_spend"].sum(), df["tv_spend"].sum()],
})
truth_table["true_incremental"] = truth_table["true_roas"] * truth_table["total_spend"]
print("\nGround Truth Table:")
display(truth_table)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df["week"], df["revenue"] / 1e6, label="Observed Revenue", alpha=0.8)
ax.plot(df["week"], df["baseline_true"] / 1e6, label="True Baseline (incl. 26-week cycle)", linestyle="--", color="orange")
ax.set_title("Revenue vs True Baseline \u2014 Single Dataset (seed=42)")
ax.set_xlabel("Week")
ax.set_ylabel("\u00a3M")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## Section 4 — Model Definitions and Single-Run Comparison

Three models differ only in baseline flexibility:
- **Model A**: No seasonality controls (under-flexible)
- **Model B**: Annual harmonics only (correct specification)
- **Model C**: Annual + bi-annual + quarterly harmonics (over-flexible)

In [ ]:
MODEL_FEATURES = {
    "A": ["meta_spend", "search_spend", "tv_spend", "trend", "promo"],
    "B": ["meta_spend", "search_spend", "tv_spend", "trend", "sin52", "cos52", "promo"],
    "C": ["meta_spend", "search_spend", "tv_spend", "trend", "sin52", "cos52", "promo", "sin26", "cos26", "sin13", "cos13"],
}

MODEL_LABELS = {
    "A": "Model A (under-flexible: no seasonality controls)",
    "B": "Model B (correct specification)",
    "C": "Model C (over-flexible: extra harmonics)",
}

def fit_model(df, model_key):
    features = MODEL_FEATURES[model_key]
    X = sm.add_constant(df[features])
    model = sm.OLS(df["revenue"], X).fit()
    return model

# Fit all three on seed=42 dataset
models = {k: fit_model(df, k) for k in ["A", "B", "C"]}

comparison = []
for k, m in models.items():
    comparison.append({
        "model": MODEL_LABELS[k],
        "meta_roas": m.params["meta_spend"],
        "search_roas": m.params["search_spend"],
        "tv_roas": m.params["tv_spend"],
        "r2": m.rsquared,
    })

true_row = {
    "model": "TRUE VALUES",
    "meta_roas": TRUE_ROAS["meta"],
    "search_roas": TRUE_ROAS["search"],
    "tv_roas": TRUE_ROAS["tv"],
    "r2": float("nan"),
}
comp_df = pd.DataFrame([true_row] + comparison)
print("Single-run ROAS comparison:")
display(comp_df)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df["week"], df["revenue"] / 1e6, label="Actual Revenue", alpha=0.7, linewidth=1.5)
colors = {"A": "tab:red", "B": "tab:green", "C": "tab:blue"}
for k in ["A", "B", "C"]:
    features = MODEL_FEATURES[k]
    X = sm.add_constant(df[features])
    fitted = models[k].predict(X)
    ax.plot(df["week"], fitted / 1e6, label=f"Model {k} fit", linestyle="--", color=colors[k], alpha=0.85)
ax.set_title("Revenue and Model Fitted Values \u2014 Single Dataset (seed=42)")
ax.set_xlabel("Week")
ax.set_ylabel("\u00a3M")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## Section 5 — Simulation Function

In [ ]:
def run_one_simulation(model_key, seed):
    df = generate_data(seed=seed)
    model = fit_model(df, model_key)
    rows = []
    for channel in ["meta", "search", "tv"]:
        col = f"{channel}_spend"
        est = model.params[col]
        true_r = TRUE_ROAS[channel]
        ci_low, ci_high = model.conf_int(alpha=0.10).loc[col]
        rows.append({
            "model": model_key,
            "seed": seed,
            "channel": channel,
            "true_roas": true_r,
            "estimated_roas": est,
            "relative_error": abs(est - true_r) / true_r,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "covered_truth": ci_low <= true_r <= ci_high,
            "r2": model.rsquared,
        })
    return rows

# Quick sanity check
test_rows = run_one_simulation("B", seed=42)
print("Sample output from run_one_simulation('B', seed=42):")
display(pd.DataFrame(test_rows))

## Section 6 — Sanity Check: Model B (Correct Specification), N=500

Before the stress loop, confirm Model B recovers true ROAS cleanly.

In [ ]:
N_SIMS_SANITY = 500
sanity_rows = []
for sim_id in range(N_SIMS_SANITY):
    seed = 20_000 + sim_id
    sanity_rows.extend(run_one_simulation("B", seed))

sanity_df = pd.DataFrame(sanity_rows)

sanity_summary = (
    sanity_df.groupby("channel")
    .agg(
        true_roas=("true_roas", "first"),
        median_est_roas=("estimated_roas", "median"),
        median_rel_error_pct=("relative_error", lambda x: x.median() * 100),
        coverage_90pct=("covered_truth", "mean"),
    )
    .reset_index()
)
print("Model B sanity check (N=500):")
display(sanity_summary)

## Section 7 — Stress Loop: All Three Models, N=200 Each

In [ ]:
N_SIMS = 200
model_keys_list = ["A", "B", "C"]
all_rows = []
for model_key in model_keys_list:
    for sim_id in range(N_SIMS):
        seed = 40_000 + model_keys_list.index(model_key) * 10_000 + sim_id
        all_rows.extend(run_one_simulation(model_key, seed))

stress_results = pd.DataFrame(all_rows)
print(f"Total rows: {len(stress_results):,}")
display(stress_results.head())

## Section 8 — Summary Table

In [ ]:
summary = (
    stress_results
    .groupby(["model", "channel"])
    .agg(
        true_roas=("true_roas", "first"),
        median_estimated_roas=("estimated_roas", "median"),
        median_relative_error_pct=("relative_error", lambda x: x.median() * 100),
        probability_over_50pct_error_pct=("relative_error", lambda x: (x > 0.50).mean() * 100),
        coverage_90pct=("covered_truth", "mean"),
        mean_r2=("r2", "mean"),
    )
    .reset_index()
)

summary["model_label"] = summary["model"].map(MODEL_LABELS)
cols = ["model_label", "channel", "true_roas", "median_estimated_roas",
        "median_relative_error_pct", "probability_over_50pct_error_pct",
        "coverage_90pct", "mean_r2"]
print("Stress test summary table:")
display(summary[cols])

## Section 9 — Main Chart: ROAS Recovery by Model

In [ ]:
summary_pivot = (
    stress_results
    .groupby(["model", "channel"])
    .agg(
        median_roas=("estimated_roas", "median"),
        true_roas=("true_roas", "first"),
    )
    .reset_index()
)

channels = ["meta", "search", "tv"]
x = np.arange(len(channels))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
bar_colors = {"A": "tab:red", "B": "tab:green", "C": "tab:blue"}

for i, model_key in enumerate(["A", "B", "C"]):
    vals = [
        summary_pivot[
            (summary_pivot["model"] == model_key) & (summary_pivot["channel"] == ch)
        ]["median_roas"].values[0]
        for ch in channels
    ]
    ax.bar(x + i * width, vals, width, label=MODEL_LABELS[model_key], color=bar_colors[model_key], alpha=0.85)

for j, ch in enumerate(channels):
    true_val = TRUE_ROAS[ch]
    ax.plot([x[j] - 0.1, x[j] + 3 * width - 0.1], [true_val, true_val], "k--", linewidth=1.5)

ax.set_xticks(x + width)
ax.set_xticklabels([c.title() for c in channels])
ax.set_title("Baseline Flexibility vs ROAS Recovery")
ax.set_ylabel("Median Estimated ROAS")
ax.legend(loc="upper left")
ax.grid(alpha=0.2, axis="y")
ax.annotate("Dashed lines = true ROAS", xy=(0.75, 0.92), xycoords="axes fraction", fontsize=9, color="black")
plt.tight_layout()
plt.show()

## Section 10 — Model Fit (R\u00b2) Chart

High R\u00b2 does not guarantee correct ROAS attribution. All three models may achieve similar R\u00b2 even when ROAS estimates diverge significantly.

In [ ]:
r2_summary = (
    stress_results.drop_duplicates(subset=["model", "seed"])
    .groupby("model")["r2"]
    .mean()
    .reset_index()
)
r2_summary["model_label"] = r2_summary["model"].map(MODEL_LABELS)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(r2_summary["model"], r2_summary["r2"],
       color=[bar_colors[m] for m in r2_summary["model"]], alpha=0.85)
ax.set_ylim(0.95, 1.0)
ax.set_title("Mean R\u00b2 by Model Across 200 Simulations")
ax.set_ylabel("Mean R\u00b2")
ax.set_xticklabels(r2_summary["model_label"], rotation=10, ha="right")
ax.grid(alpha=0.2, axis="y")
plt.tight_layout()
plt.show()
print("Note: All three models achieve very similar R\u00b2 \u2014 fit quality alone cannot distinguish correct from incorrect specification.")

## Section 11 — Distribution of Estimated Meta ROAS Across Simulations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 5), sharey=False)
for i, model_key in enumerate(["A", "B", "C"]):
    data = stress_results[
        (stress_results["model"] == model_key) & (stress_results["channel"] == "meta")
    ]["estimated_roas"]
    axes[i].hist(data, bins=30, orientation="vertical",
                 color=bar_colors[model_key], edgecolor="white", linewidth=0.5, alpha=0.85)
    axes[i].axvline(TRUE_ROAS["meta"], color="red", linestyle="--", linewidth=1.5, label=f"True ROAS = {TRUE_ROAS['meta']}")
    axes[i].set_title(f"Model {model_key}")
    axes[i].set_xlabel("Estimated Meta ROAS")
    if i == 0:
        axes[i].set_ylabel("Count")
    axes[i].legend(fontsize=8)

plt.suptitle("Distribution of Estimated Meta ROAS Across 200 Simulations", fontsize=13)
plt.tight_layout()
plt.show()

## Section 12 — Interpretation

> "Baseline specification is part of the attribution model, not just a nuisance component."

**Model A (under-flexible):** No 26-week harmonic is included. When the medium-frequency demand cycle co-moves with media spend (even weakly, due to RHO=0.10), the model has no dedicated term to absorb it. Some of that variation is incorrectly credited to the channel coefficients, inflating ROAS estimates.

**Model C (over-flexible):** The extra harmonics (sin26, cos26, sin13, cos13) can partially absorb not just the genuine baseline variation but also some of the media-driven revenue variation \u2014 particularly when spend patterns happen to correlate with these harmonics. This attenuates ROAS estimates downward.

**Model B (correct specification):** Includes sin52/cos52 to capture the true annual cycle, plus a linear trend. This matches the DGP structure and recovers ROAS most accurately.

**The real-world problem:** In practice, the true baseline structure is unknown. You cannot directly test whether your baseline is correctly specified, and R\u00b2 gives you no signal \u2014 all three models achieve similar fit. This makes baseline misspecification a latent, hard-to-detect source of ROAS bias.

## Section 13 — LinkedIn Takeaway

I built the same dataset three times. Then I fitted three models with different baseline assumptions.

Model A had no seasonality controls. Model B had the right ones. Model C had too many.

The results:
- Model A **overestimated** media ROAS because a 26-week demand cycle bled into the channel coefficients.
- Model C **underestimated** media ROAS because the flexible baseline absorbed some of the genuine media effect.
- Model B \u2014 the correct specification \u2014 recovered ROAS accurately.

All three models had similar R\u00b2. You cannot tell them apart from fit metrics alone.

This is the baseline flexibility problem in MMM. The baseline is not a nuisance to be swept aside. It is part of the attribution model. Get it wrong \u2014 in either direction \u2014 and your ROAS is wrong.

The tricky part: in real campaigns, you don't know the true baseline. You have to make modelling choices without a ground truth to check against. That's why reproducible stress tests with known DGPs matter.

---
*Part of the MMM Stress Test Series \u2014 where we deliberately break models to understand when they can be trusted.*

In [ ]:
stress_results.to_csv("mmm_stress_test_07_raw_results.csv", index=False)
summary[cols].to_csv("mmm_stress_test_07_summary.csv", index=False)
print("Saved:")
print("  mmm_stress_test_07_raw_results.csv")
print("  mmm_stress_test_07_summary.csv")

## End of Notebook

---